[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ScriptsRemote/Transcend_SummerSchool/blob/main/Day4_LULC_and_Climate_with_RS/01_classification_fundamentals_and_algorithms_es.ipynb)

# Día 4 - Uso y Cobertura del Suelo y Clima con Teledetección
## Sesión 1: Fundamentos y algoritmos de clasificación

| | |
|---|---|
| **Fecha** | 2026-10-15 (jueves) |
| **Horario** | 09:00 - 10:30 |
| **Entidad** | UoM |
| **Instructores** | Christhian, Sebastian integra WETSAT |

### Contenido

> Qué es el aprendizaje automático; clasificación de imágenes supervisada y no supervisada; K-Means, PCA,
> CART, SVM y Random Forest; muestras de entrenamiento vs prueba; matriz de confusión, exactitud del productor y
> del consumidor; importancia de las variables.

### Objetivos de aprendizaje

- [ ] Explicar qué es el aprendizaje automático y la diferencia entre el aprendizaje **supervisado** y el
      **no supervisado**
- [ ] Describir cómo funcionan los principales clasificadores: distancia mínima, máxima verosimilitud, KNN, SVM,
      árboles de decisión (CART) y Random Forest
- [ ] Ejecutar una clasificación no supervisada (**K-Means**) en Google Earth Engine
- [ ] Entrenar **CART, SVM y Random Forest** con muestras recolectadas en el recolector de muestras de
      TRANSCEND, y comparar los mapas
- [ ] Evaluar una clasificación con una **matriz de confusión**, la exactitud global, la exactitud del productor y
      del consumidor, e interpretar la **importancia de las variables** del Random Forest

### Datos necesarios

- Landsat 8 y 9 Colección 2 Nivel 2 y NASADEM (catálogo de Earth Engine)
- Las muestras de entrenamiento (GeoJSON) exportadas desde el
  [TRANSCEND Sample Collector](https://ee-christhianscunha.projects.earthengine.app/view/transcend-samplecollector)

### Lo que ya vimos

En el Día 3, sesión 4, se construyó el **cubo de datos**: Landsat 8/9 con máscara de nubes, factor de escala y los
índices NDVI, MSAVI, EVI, NDWI y NDMI como bandas adicionales. Esta sesión usa ese cubo como
entrada de los clasificadores.

---

## 1.1 - Teoría: ¿qué es el aprendizaje automático?

<div style="display:flex; gap:20px; align-items:center">

<div style="width:300px">

![image-2.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c02_image-2.png)

</div>

<div style="width:450px">

![robot](https://kokorokids.app/wp-content/uploads/2018/07/robot_stacking_blocks.gif)

</div>

</div>

Según Géron (2019) y Sousa (2021), el **aprendizaje automático (Machine Learning, ML)** es un campo de la Inteligencia
Artificial cuyo objetivo es programar computadoras para que **aprendan a partir de datos**.

Para VanderPlas (2016), el aprendizaje automático consiste en construir **modelos matemáticos que nos ayudan a
entender los datos**. El "aprendizaje" ocurre cuando damos a estos modelos **parámetros ajustables**
que se adaptan a los datos observados - en este sentido, el programa está "aprendiendo" de los datos.

El aprendizaje automático es una capa dentro del campo más amplio de la Inteligencia Artificial; el Deep
Learning (aprendizaje profundo) es una familia de métodos de ML basados en redes neuronales con muchas capas, y la IA
Generativa es una rama del Deep Learning.

Jordan & Mitchell (2015) describen el aprendizaje automático como la pregunta de cómo construir sistemas
computacionales que **mejoran automáticamente a través de la experiencia**. Janiesch et al. (2021) agregan que,
en lugar de seguir reglas escritas por un programador, los algoritmos de ML **derivan las reglas a partir de
ejemplos** - y el Deep Learning es el subconjunto del ML que usa redes neuronales artificiales con
muchas capas. En teledetección, algoritmos como SVM, Random Forest y los árboles de decisión se han
convertido en herramientas estándar para la clasificación de imágenes porque manejan muchas bandas y relaciones
no lineales mejor que los clasificadores estadísticos tradicionales (Maxwell et al., 2018).

<div style="width:650px">

![image.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c02_image.png)

</div>

*Fuente: Zhuhadar & Lytras (2023), https://doi.org/10.3390/su151813484*

### Vocabulario básico

- **Dataset (conjunto de datos)** - el conjunto de datos. Cada dato es una **observación** individual (en nuestro
  caso, un **píxel** o un **punto de muestra**).
- **Features (variables)** - cada observación se describe con un número fijo de características
  (variables). Pueden ser continuas, discretas o categóricas. En teledetección las variables
  son las **bandas** y los **índices** de la imagen: Blue, Red, NIR, NDVI, elevación...
- **Variables de entrada y salida** - ejemplo: un estudio agrícola mide (a) la intensidad de la luz
  y (b) las horas de luz por día, y quiere predecir la **producción de frutos**. (a) y (b) son
  las *entradas*; la producción de frutos es la *salida*. En la clasificación de imágenes las entradas son los
  valores de las bandas del píxel y la salida es su **clase** (agua, suelo, urbano...).

### Los pasos de un modelo predictivo (pipeline)

0. **Entender el problema** - ¿qué pregunta estamos respondiendo?
1. **Entender los datos** - distribución, valores atípicos, estructurados o no.
2. **Análisis exploratorio** - ¿tenemos todos los "ingredientes"?
3. **Limpieza y preprocesamiento** - eliminar valores atípicos, sin datos, nubes (¡Día 3!).
4. **Entrenar el modelo** - dividir las muestras en **entrenamiento** y **prueba** (p. ej. 70% / 30%).
5. **Probar y evaluar** - métricas de exactitud y validación.
6. **Implementar** - aplicar el modelo a toda la imagen y usar el mapa.

![Pipeline de ML](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c03_s07_1.png)

*Fuente: Belsare & Singh (2022)*

### ¿Cómo aprenden las máquinas?

<div style="width:500px">

![image.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c04_image.png)

</div>


Según los datos que damos y el problema que queremos resolver, las máquinas aprenden de tres formas:

- **Aprendizaje supervisado** - los datos tienen etiquetas (se conoce la respuesta correcta).
- **Aprendizaje no supervisado** - los datos no tienen etiquetas; el algoritmo encuentra la estructura por sí solo.
- **Aprendizaje por refuerzo** - un agente aprende por ensayo y error, con recompensas y penalizaciones.

La clasificación de imágenes usa las dos primeras.

![Tipos de aprendizaje automático](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c04_s06_1.png)

### Aprendizaje no supervisado

- Descubre **patrones y estructuras ocultas** en los datos.
- Trabaja con datos **no etiquetados** (sin salidas correctas).
- Familias principales: **agrupamiento (clustering)**, **reducción de dimensionalidad** y análisis jerárquico.

El aprendizaje no supervisado modela las variables de un conjunto de datos sin referencia a ninguna etiqueta - a
menudo se describe como **"dejar que el conjunto de datos hable por sí mismo"** (VanderPlas, 2016).

![Aprendizaje no supervisado: datos sin etiquetas divididos en grupos](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c05_s15_1.png)

#### K-Means

K-Means divide los datos en **k grupos (clústeres)**. Comienza con k centros aleatorios, asigna
cada píxel al centro más cercano, recalcula los centros como la media de sus píxeles, y
repite hasta que los centros dejan de moverse. El analista solo elige **k**; el significado de cada
clúster (¿agua? ¿suelo?) se decide **después**, mirando el mapa.

![Antes y después de K-Means](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c05_s16_2.png)

En una imagen, K-Means agrupa píxeles que están cerca unos de otros en el **espacio de atributos**
(valores de las bandas), no en el mapa. La elección de la distancia (euclidiana, Manhattan) y de k
cambia el resultado:

![K-Means en imágenes MODIS con distancia euclidiana](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c06_s17_1.jpg)
![K-Means con k=3 y k=6 y el espacio de atributos](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c06_s18_1.jpg)

*Fuentes: Rozanov (2024); Zanotta et al. (2019)*

#### Análisis de Componentes Principales (PCA)

Propuesto originalmente por Pearson (1901), el **PCA** es una herramienta exploratoria que usa una transformación
ortogonal para convertir un conjunto de variables posiblemente **correlacionadas** en nuevas variables **no correlacionadas**,
las **componentes principales** (Zanotta et al., 2019).

**La idea en palabras simples.** En una imagen satelital, muchas bandas llevan casi la misma
información. Un píxel brillante (arena, un techo) es brillante en Blue, Green y Red al mismo tiempo;
un píxel oscuro (agua, sombra) es oscuro en todas ellas. Al graficar una banda contra otra, los
píxeles forman una nube alargada a lo largo de una diagonal - las bandas están **correlacionadas**, y gran parte de
lo que dicen se **repite**.

El PCA **rota los ejes** de esta nube:

- **PC1** sigue la dirección más larga de la nube - capta lo que todas las bandas tienen en
  común, normalmente el **brillo general** del píxel;
- **PC2** es perpendicular a PC1 y capta las **diferencias entre bandas** - en una
  imagen Landsat que incluye el NIR, a menudo resalta la **vegetación** (NIR alto, Red bajo);
- las componentes siguientes contienen cada vez menos información, y las últimas son sobre todo
  **ruido**.

![image.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c07_image.png)

*Ejemplo ilustrativo con dos bandas (datos sintéticos). Elaborado por los autores.*

**Por qué es útil en teledetección:**

- **Reducir datos** - 6 o 10 bandas a menudo se pueden resumir en 2 o 3 componentes con más del
  90% de la información, lo que hace más rápida la clasificación;
- **Eliminar ruido** - descartar las últimas componentes limpia la imagen;
- **Resaltar contrastes** - las composiciones en color de PC1, PC2 y PC3 revelan diferencias que son
  difíciles de ver en las bandas originales, por ejemplo en el **mapeo geológico**:

![Composición PCA para mapeo geológico](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c07_s19_1.jpg)

*Fuente: Jiang et al. (2013), Remote Sens. 5, 3156-3171, doi:10.3390/rs5073156*

> **Atención:** las componentes principales **no tienen un significado físico fijo** - se
> calculan a partir de cada imagen, así que la PC2 de una escena no es la misma que la PC2 de otra.

### Aprendizaje supervisado

- Relaciona **entradas con salidas predefinidas** usando datos **etiquetados**.
- Se usa para **clasificación** (una categoría) y **regresión** (un número): regresión lineal,
  K vecinos más cercanos (KNN), máquinas de vectores de soporte (SVM), Random Forest y CART.

El aprendizaje supervisado modela la relación entre las variables medidas y una **etiqueta**;
una vez ajustado el modelo, se puede usar para etiquetar **datos nuevos y desconocidos** (VanderPlas, 2016).
En el entrenamiento le damos al modelo las variables de entrada **junto con la salida esperada** - el
modelo aprende de estos pares entrada-salida.

![Aprendizaje supervisado: datos etiquetados](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c08_s20_1.png)
![Aprender a partir de ejemplos](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c08_s21_1.jpg)

### Parámetros, hiperparámetros e importancia de las variables

- **Parámetros** - los modelos de ML son funciones matemáticas con valores que se **ajustan
  durante el entrenamiento** para alcanzar un mejor desempeño. Se estiman **a partir de los datos**.
- **Hiperparámetros** - valores elegidos **por el analista** antes del entrenamiento (número de árboles,
  profundidad de un árbol, k de K-Means...). No se aprenden de los datos, pero afectan el
  resultado. Una práctica común es probar varios valores y quedarse con el mejor:

![Exactitud según el número de árboles](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c09_s22_1.png)

*Fuente: Gandhi (2021)*

- **Importancia de las variables** - muchos clasificadores en GEE tienen un método `explain()`. El clasificador
  le da a cada variable de entrada una **puntuación** de cuán útil fue para predecir la clase correcta.
  La calcularemos al final de este notebook.

![Gráfico de importancia de las variables](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c09_s23_1.png)

*Fuente: Gandhi (2021)*

---

## 1.2 - Teoría: clasificación de imágenes

La **clasificación** es el proceso de asignar una **etiqueta** a datos descritos por un conjunto de
atributos. En las imágenes de teledetección esto significa decidir, **para cada píxel**, qué categoría hay
en el terreno - agua, suelo, bosque... - normalmente a partir de sus **atributos espectrales**, el valor
en cada banda (Jensen, 2005; Zanotta et al., 2019).

| | |
|:---:|:---:|
| ![image.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c10_image.png) | ![image-2.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c10_image-2.png) |
| ![image-3.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c10_image-3.png) | ![image-4.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c10_image-4.png) |

Cada píxel es un punto en el **espacio de atributos** (un eje por banda). Los píxeles de la misma clase
forman nubes de puntos; el clasificador traza los **límites** entre estas nubes.

![Clases en el espacio de atributos y sus histogramas](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c10_s26_1.png)

*Fuente: Zanotta et al. (2019)*

### Ejemplos de clasificadores

**Distancia mínima (euclidiana)** - usa solo la **media** de cada clase en el espacio de atributos.
Un píxel se asigna a la clase cuya media es la más cercana.

**Máxima verosimilitud (MaxVer)** - modela cada clase como una distribución gaussiana. La clase
ganadora es la que tiene la **mayor probabilidad conjunta** (altura de la curva gaussiana) en
todas las bandas.

![Clasificador de distancia mínima](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c11_s26_2.png)

*Fuente: Zanotta et al. (2019)*

**KNN - k vecinos más cercanos** - uno de los algoritmos más simples e intuitivos. Cada nueva
muestra recibe la **clase mayoritaria entre sus k vecinos más cercanos** en el espacio de atributos.

![KNN antes y después](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c11_s27_2.png)

**SVM - Máquina de vectores de soporte** - busca un **hiperplano** que separe las clases,
con el **margen máximo**: el límite que está lo más lejos posible de las muestras de entrenamiento
más cercanas (los *vectores de soporte*). Con una formulación no lineal (kernel, p. ej. RBF) puede trazar
límites curvos y aprender funciones complejas. El SVM es sensible a la **escala** de las
variables, por lo que las bandas deben estandarizarse.

![Clases separables lineal y no linealmente](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c12_s27_1.png)

*Fuente: Zanotta et al. (2019)*

**Árbol de decisión (CART)** - divide los atributos de entrada de forma **secuencial** en subespacios que
aíslan las clases. El problema se descompone en decisiones simples - preguntas objetivas como
*"¿es NDVI > 0.5?"* - que se resuelven de forma recursiva hasta que cada hoja contiene una clase.

![Árbol de decisión con reglas de NDVI y SWIR](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c12_s28_1.png)

**Random Forest** - propuesto por Breiman (2001), muy usado en aprendizaje automático y con muy buenos
resultados en teledetección. **Combina muchos árboles de decisión**, cada uno entrenado con un subconjunto
aleatorio diferente de las muestras y de las variables. La clase final se decide por **voto
mayoritario**. Esto lo hace robusto frente al ruido y al sobreajuste, y además entrega la importancia de
cada variable.

![Random Forest: muchos árboles y voto mayoritario](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c13_s28_2.png)

*Fuente: Dang (2020)*

---

## 1.3 - Teoría: evaluación de la exactitud

Las muestras se **dividen** en un conjunto de **entrenamiento** (usado para enseñar al clasificador) y un
conjunto de **prueba** (que se mantiene aparte y el clasificador nunca ve). Las etiquetas de prueba se comparan con
las clases predichas.

![Flujo de entrenamiento y prueba](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c14_s29_1.png)

*Fuente: Escovedo & Koshiyama (2020)*

Para cada clase contamos las muestras detectadas correctamente y las muestras asignadas erróneamente a ella.
Estos resultados se organizan en una **matriz de confusión** (matriz de error): en GEE las **filas son la
referencia** (muestras de prueba) y las **columnas son la clasificación**.

| Métrica | Pregunta que responde | Cómo se calcula |
|---|---|---|
| **Exactitud global (OA)** | ¿Qué porcentaje de las muestras es correcto? | suma de la diagonal / total |
| **Exactitud del consumidor (del usuario)** | Si el **mapa** dice "agua", ¿qué tan probable es que realmente sea agua? | diagonal / total de la **columna** |
| **Exactitud del productor** | De las muestras reales de agua, ¿cuántas se mapearon como agua? | diagonal / total de la **fila** |
| **Error de comisión** | Píxeles **incluidos erróneamente** en una clase | 1 - exactitud del consumidor |
| **Error de omisión** | Píxeles de una clase que **quedaron fuera** | 1 - exactitud del productor |
| **Kappa** | Concordancia más allá de la que daría el azar | a partir de toda la matriz |

**Autocorrelación espacial.** Los píxeles vecinos son muy parecidos. Si un píxel de entrenamiento y un
píxel de prueba vienen de la misma parcela, la prueba es demasiado fácil y la exactitud queda **inflada**.
Por eso dividimos las **geometrías de las muestras** (puntos o polígonos) en entrenamiento y prueba
**antes** de extraer los píxeles.

---

## 1.4 - Práctica guiada: configuración y cubo de datos

El área de estudio tiene dos partes a orillas del **Lago Titicaca**: una cerca de **Puno** (Perú) y una
del lado boliviano. Usamos solo cinco clases:

**Agua, Vegetación, Suelo, Urbano y Agricultura**

In [ ]:
# Librerías (descomente la línea de pip en un entorno nuevo, p. ej. Google Colab)
#%pip install -q earthengine-api geemap

import os
import ee                          # API de Google Earth Engine
import geemap                      # mapas interactivos para Earth Engine
import pandas as pd                # tablas (DataFrames)
import matplotlib.pyplot as plt    # gráficos
import seaborn as sns              # mapa de calor de la matriz de confusión

In [ ]:
# Autenticar e inicializar Earth Engine
# >>> REEMPLAZAR por el id de su propio proyecto de Cloud <<<
EE_PROJECT = "ee-christhianscunha"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

In [ ]:
# Área de estudio: dos rectángulos a orillas del Lago Titicaca
REGION =  ee.Geometry.Polygon(
        [[[-69.1010932684908, -16.14842858486395],
          [-69.1010932684908, -16.17175891406604],
          [-69.06075284490682, -16.17175891406604],
          [-69.06075284490682, -16.14842858486395]]], None, False);
Map = geemap.Map()
Map.centerObject(REGION, 12)
Map.add_basemap("SATELLITE")
Map.addLayer(REGION, {"color": "red"}, "Study area")
Map

### El cubo de datos (del Día 3)

Las mismas tres funciones del Día 3: máscara de nubes, factor de escala e índices. Con ellas construimos una
**composición de mediana** de la época seca (mayo-septiembre), cuando el cielo sobre el altiplano está
despejado. La mediana de muchas fechas elimina las nubes y el ruido restantes.

In [ ]:
def mask_clouds(image):
    # Bits de QA_PIXEL 1 (nube dilatada), 2 (cirros), 3 (nube) y 4 (sombra de nube)
    cloud_bits = (1 << 1) | (1 << 2) | (1 << 3) | (1 << 4)
    clear = image.select("QA_PIXEL").bitwiseAnd(cloud_bits).eq(0)
    return image.updateMask(clear)


def scale_and_rename(image):
    # Factor de escala de la Colección 2 -> reflectancia de superficie, y nombres de bandas legibles
    optical = image.select(["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"])
    optical = optical.multiply(0.0000275).add(-0.2)
    return optical.rename(["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"])


def add_indices(image):
    ndvi = image.normalizedDifference(["NIR", "Red"]).rename("NDVI")
    ndwi = image.normalizedDifference(["Green", "NIR"]).rename("NDWI")
    ndmi = image.normalizedDifference(["NIR", "SWIR1"]).rename("NDMI")
    evi = image.expression(
        "2.5 * (NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1)",
        {"NIR": image.select("NIR"), "RED": image.select("Red"), "BLUE": image.select("Blue")},
    ).rename("EVI")
    msavi = image.expression(
        "(2 * NIR + 1 - sqrt((2 * NIR + 1) ** 2 - 8 * (NIR - RED))) / 2",
        {"NIR": image.select("NIR"), "RED": image.select("Red")},
    ).rename("MSAVI")
    return image.addBands([ndvi, msavi, evi, ndwi, ndmi])

In [ ]:
# Landsat 8 + 9, época seca 2024, sobre el área de estudio
collection = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
                .filterBounds(REGION)
                .filterDate("2024-05-01", "2024-09-30")
                .filter(ee.Filter.lt("CLOUD_COVER", 30))
                .map(mask_clouds)
                .map(scale_and_rename)
                .map(add_indices))

# Composición de mediana: un valor por píxel y banda
composite = collection.median()

print("Images used in the composite:", collection.size().getInfo())

### Agregar elevación y pendiente (NASADEM)

Algunas clases son difíciles de separar solo con información espectral - por ejemplo la agricultura
en terrazas planas frente a la vegetación natural en colinas empinadas. Agregar la **elevación** y la **pendiente**
del modelo digital de elevación NASADEM (30 m) le da al clasificador esta información adicional.

In [ ]:
# Elevación (m) y pendiente (grados) de NASADEM
elevation = ee.Image("NASA/NASADEM_HGT/001").select("elevation")
slope = ee.Terrain.slope(elevation).rename("slope")

# La imagen final: 6 bandas + 5 índices + elevación + pendiente
image = composite.addBands([elevation, slope]).toFloat()

FEATURES = image.bandNames().getInfo()
print("Features:", FEATURES)

### Estandarizar las variables

La reflectancia va de 0 a ~0.5, el NDVI de -1 a 1, la elevación de ~3800 a ~4500 m. Los métodos
basados en **distancia** (K-Means, SVM) estarían dominados por la elevación solo porque sus números
son mayores. **Estandarizamos** cada banda: restamos su media y dividimos por su desviación
estándar, de modo que todas las variables tengan media 0 y desviación estándar 1.

CART y Random Forest no lo necesitan, pero usar la misma imagen estandarizada para todos los
métodos mantiene la comparación justa.

In [ ]:
# Media y desviación estándar de cada banda sobre el área de estudio
mean = image.reduceRegion(ee.Reducer.mean(), REGION, 30, maxPixels=1e9)
std = image.reduceRegion(ee.Reducer.stdDev(), REGION, 30, maxPixels=1e9)

# (valor - media) / desviación estándar, banda por banda
image_std = image.subtract(mean.toImage(FEATURES)).divide(std.toImage(FEATURES))

# Visualización de la imagen real (no estandarizada)
vis_rgb = {"bands": ["Red", "Green", "Blue"], "min": 0.0, "max": 0.3}
vis_false = {"bands": ["NIR", "Red", "Green"], "min": 0.0, "max": 0.4}

Map = geemap.Map()
Map.centerObject(REGION,14)
Map.addLayer(image.clip(REGION), vis_rgb, "RGB composite")
Map.addLayer(image.clip(REGION), vis_false, "False colour (NIR-R-G)", False)
Map

---

## 1.5 - Clasificación no supervisada: K-Means

Tres pasos:

1. tomar una **muestra aleatoria** de píxeles de la imagen (¡sin etiquetas!);
2. **entrenar** el agrupador, eligiendo solo el número de clústeres (k = 5, uno por cada clase que
   esperamos);
3. aplicarlo a toda la imagen con `cluster()`.

El resultado es un mapa de **clústeres numerados de 0 a 4**. Los números **no tienen significado** hasta
que los interpretamos comparándolos con la imagen RGB.

In [ ]:
# 1. Muestra aleatoria de 5000 píxeles del área de estudio
kmeans_sample = image_std.sample(region=REGION, scale=30, numPixels=5000, seed=42)

# 2. Entrenar K-Means con k = 5 clústeres
kmeans = ee.Clusterer.wekaKMeans(5).train(kmeans_sample)

# 3. Aplicarlo a cada píxel
clusters = image_std.cluster(kmeans).clip(REGION)

Map = geemap.Map()
Map.centerObject(REGION, 14)
Map.addLayer(image.clip(REGION), vis_rgb, "RGB composite")
Map.addLayer(clusters.randomVisualizer(), {}, "K-Means (5 clusters)")
Map

**Pregunta.** Active y desactive la capa de K-Means. ¿Qué clúster es agua? ¿Hay un clúster que
mezcla urbano y suelo? K-Means encuentra los grupos que son **espectralmente** diferentes, que no
siempre son las clases que **nosotros** queremos. Esa es la motivación de la clasificación supervisada.

---

## 1.6 - Clasificación supervisada

### Cargar las muestras

Las muestras se recolectaron en el
[TRANSCEND Sample Collector](https://ee-christhianscunha.projects.earthengine.app/view/transcend-samplecollector)
y se exportaron como **GeoJSON**. Cada muestra guarda el **código de la clase** (un número) y el
**nombre de la clase**.

![image.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day4/01_c29_image.png)

1. Suba el GeoJSON a la carpeta de trabajo (en Colab: el ícono de carpeta a la izquierda -> subir).
2. Verifique el nombre del archivo y de los dos atributos de abajo.
3. `geemap.coreutils.geojson_to_ee()` convierte el archivo en una `ee.FeatureCollection` que Earth
   Engine puede usar.

> Si no se encuentra el archivo, la celda siguiente crea **muestras de demostración** a partir del mapa
> ESA WorldCover, para que el notebook igual se ejecute. Use sus propias muestras para el ejercicio.

In [ ]:
# >>> Ajuste a su archivo <<<
SAMPLES_FILE = "/content/drive/MyDrive/SummerSchool/Assets/copacabana_samples.geojson"   # GeoJSON exportado desde el Sample Collector
CLASS_PROPERTY = "class"                     # atributo con el código de la clase (número)
NAME_PROPERTY = "class_name"                       # atributo con el nombre de la clase

if os.path.exists(SAMPLES_FILE):
    # GeoJSON -> ee.FeatureCollection
    samples = geemap.coreutils.geojson_to_ee(SAMPLES_FILE)
else:
    # Muestras de demostración: puntos aleatorios por clase a partir de ESA WorldCover 2021
    # Códigos de WorldCover -> nuestras clases: 80 agua=1 | 10,20,30 vegetación=2 | 60 suelo=3 | 50 urbano=4 | 40 cultivos=5
    print("GeoJSON not found - using demonstration samples from ESA WorldCover")
    worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
    reference = worldcover.remap([80, 10, 20, 30, 60, 50, 40], [1, 2, 2, 2, 3, 4, 5]).rename(CLASS_PROPERTY)
    samples = reference.stratifiedSample(numPoints=60, classBand=CLASS_PROPERTY, region=REGION,
                                         scale=30, seed=42, geometries=True)
    names = ee.Dictionary({"1": "Water", "2": "Vegetation", "3": "Soil", "4": "Urban", "5": "Agriculture"})
    samples = samples.map(lambda f: f.set(NAME_PROPERTY, names.get(ee.Number(f.get(CLASS_PROPERTY)).int().format())))

print("Number of samples:", samples.size().getInfo())
samples.limit(5)

In [ ]:
# Las muestras como DataFrame, para verificar los atributos
df_samples = geemap.ee_to_df(samples)

# Códigos y nombres de las clases, en el orden del código
classes = (df_samples[[CLASS_PROPERTY, NAME_PROPERTY]]
           .drop_duplicates()
           .sort_values(CLASS_PROPERTY)
           .reset_index(drop=True))

CLASS_CODES = classes[CLASS_PROPERTY].astype(int).tolist()
CLASS_NAMES = classes[NAME_PROPERTY].tolist()

# Un color por clase, en el mismo orden que los códigos
# (agua, vegetación, suelo, urbano, agricultura - cámbielo si sus códigos están en otro orden)
PALETTE = ["#2c7bb6", "#a21616", "#d8b365", "#2ae01a", "#cc7218f3"]

classes["colour"] = PALETTE[:len(classes)]
classes

### Muestras de entrenamiento y de prueba

Dividimos las **muestras (geometrías)** de forma aleatoria: **70% para entrenamiento** y **30% para prueba**.
Solo entonces extraemos los valores de los píxeles con `sampleRegions`: para cada muestra, el valor de
las 13 variables y el código de la clase. Esto produce una tabla - exactamente el *dataset* de la
teoría: las filas son observaciones, las columnas son las variables y la etiqueta.

In [ ]:
# Número aleatorio entre 0 y 1 para cada muestra, luego división 70 / 30
samples = samples.randomColumn("random", seed=42)
train_samples = samples.filter(ee.Filter.lt("random", 0.7))
test_samples = samples.filter(ee.Filter.gte("random", 0.7))

# Extraer los valores de los píxeles de las variables bajo cada muestra
train = image_std.sampleRegions(collection=train_samples, properties=[CLASS_PROPERTY],
                                scale=30, tileScale=4)
test = image_std.sampleRegions(collection=test_samples, properties=[CLASS_PROPERTY],
                               scale=30, tileScale=4)

# Número de píxeles por clase en cada conjunto
pd.DataFrame({"train": train.aggregate_histogram(CLASS_PROPERTY).getInfo(),
              "test": test.aggregate_histogram(CLASS_PROPERTY).getInfo()})

In [ ]:
# Un vistazo a la tabla de entrenamiento: una fila por píxel, una columna por variable (estandarizada)
geemap.ee_to_df(train.limit(5))

In [ ]:
# Mostrar las muestras de entrenamiento y de prueba en el mapa
Map = geemap.Map()
Map.centerObject(REGION, 9)
Map.addLayer(image.clip(REGION), vis_rgb, "RGB composite")
Map.addLayer(train_samples, {"color": "yellow"}, "Training samples")
Map.addLayer(test_samples, {"color": "magenta"}, "Test samples")
Map

### Autocorrelación espacial: mantener separados el entrenamiento y la prueba

Observe el mapa de arriba: algunos puntos **amarillos** (entrenamiento) están justo al lado de puntos **magenta** (prueba).
Los píxeles vecinos son casi idénticos, así que probar en un píxel cuyo vecino se
usó en el entrenamiento es como darles a los estudiantes las respuestas del examen por adelantado - la exactitud
parece mejor de lo que realmente es.

La [guía de clasificación de Earth Engine](https://developers.google.com/earth-engine/guides/classification)
propone una solución simple:

> *"Una forma de excluir las muestras que podrían estar correlacionadas de esta manera es eliminar las muestras
> que están a cierta distancia de cualquier otra muestra. Esto se puede lograr con una
> unión espacial."*

Mantenemos las muestras de **prueba** tal como están y **eliminamos las muestras de entrenamiento que están a menos
de `MIN_DISTANCE` metros de cualquier muestra de prueba**:

- `ee.Filter.withinDistance` - la regla "a menos de X metros";
- `ee.Join.inverted()` - conserva solo las muestras de entrenamiento que **no** cumplen esa regla.

In [ ]:
MIN_DISTANCE = 110   # metros - pruebe 50, 100 o 300 y vea cuántas muestras quedan

# Regla: dos geometrías a menos de MIN_DISTANCE
too_close = ee.Filter.withinDistance(distance=MIN_DISTANCE, leftField=".geo",
                                     rightField=".geo", maxError=10)

# Muestras de entrenamiento lejos de toda muestra de prueba (se conservan) y cerca de alguna (se eliminan)
train_samples_far = ee.Join.inverted().apply(train_samples, test_samples, too_close)
train_samples_removed = ee.Join.simple().apply(train_samples, test_samples, too_close)

print("Training samples before:", train_samples.size().getInfo())
print("Training samples after :", train_samples_far.size().getInfo())

In [ ]:
# El mismo mapa, ahora sin las muestras de entrenamiento que estaban demasiado cerca de una muestra de prueba
Map = geemap.Map()
Map.centerObject(REGION, 9)
Map.addLayer(image.clip(REGION), vis_rgb, "RGB composite")
Map.addLayer(train_samples_far, {"color": "#ffff00"}, "Training samples (kept)")
Map.addLayer(test_samples, {"color": "#ff00ff"}, "Test samples")
Map.addLayer(train_samples_removed, {"color": "#ff0000"}, "Training samples removed (too close)")
Map.add_legend(title="Samples", legend_dict={"Training (kept)": "#ffff00", "Test": "#ff00ff",
                                             "Removed (too close)": "#ff0000"})
Map

Compare los dos mapas: los puntos **rojos** son las muestras de entrenamiento que estaban demasiado cerca de una
muestra de prueba. De aquí en adelante entrenamos con las muestras **separadas espacialmente**. Las muestras de prueba
no cambian.

> Elija una distancia que todavía deje a **cada clase** con suficientes muestras de entrenamiento - verifique la
> tabla de abajo. Una clase sin muestras no se puede predecir.

In [ ]:
# Extraer de nuevo los valores de los píxeles, ahora solo para las muestras de entrenamiento separadas
train = image_std.sampleRegions(collection=train_samples_far, properties=[CLASS_PROPERTY],
                                scale=30, tileScale=4)

# Número de píxeles por clase en cada conjunto
pd.DataFrame({"train": train.aggregate_histogram(CLASS_PROPERTY).getInfo(),
              "test": test.aggregate_histogram(CLASS_PROPERTY).getInfo()})

### Una función para mostrar cada clasificación

Los tres clasificadores siguen los **mismos tres pasos**: crear el clasificador, entrenarlo con `train()` usando
la tabla de entrenamiento, y clasificar la imagen con `classify()`. Para mostrar los tres mapas de la misma forma, escribimos
el mapa una sola vez como una función.

In [ ]:
# Visualización de las clases
vis_classes = {"min": min(CLASS_CODES), "max": max(CLASS_CODES), "palette": PALETTE}


def show_classification(classified, title):
    m = geemap.Map()
    m.centerObject(REGION, 14)
    m.addLayer(image.clip(REGION), vis_rgb, "RGB composite")
    m.addLayer(classified, vis_classes, title)
    m.add_legend(title=title, legend_dict=dict(zip(CLASS_NAMES, PALETTE)))
    return m

### CART - árbol de decisión

`ee.Classifier.smileCart()` construye un solo árbol de decisión con reglas sobre las variables.

In [ ]:
# Crear y entrenar el clasificador
cart = ee.Classifier.smileCart().train(
    features=train, classProperty=CLASS_PROPERTY, inputProperties=FEATURES)

# Clasificar la imagen
classified_cart = image_std.classify(cart).clip(REGION)

show_classification(classified_cart, "CART")

### SVM - Máquina de vectores de soporte

`ee.Classifier.libsvm()` con un **kernel RBF** (límites curvos). Dos hiperparámetros:

- `gamma` - hasta dónde llega la influencia de cada muestra (alto = límites muy locales);
- `cost` - cuánto penaliza el modelo los errores de entrenamiento (alto = menos tolerante, riesgo de
  sobreajuste).

In [ ]:
# Crear y entrenar el clasificador
svm = ee.Classifier.libsvm(kernelType="RBF", gamma=0.1, cost=10).train(
    features=train, classProperty=CLASS_PROPERTY, inputProperties=FEATURES)

# Clasificar la imagen
classified_svm = image_std.classify(svm).clip(REGION)

show_classification(classified_svm, "SVM")

### Random Forest

`ee.Classifier.smileRandomForest()` con **100 árboles**. La `seed` fija las elecciones aleatorias,
para que todos en la clase obtengan el mismo resultado.

In [ ]:
# Crear y entrenar el clasificador
rf = ee.Classifier.smileRandomForest(numberOfTrees=100, seed=42).train(
    features=train, classProperty=CLASS_PROPERTY, inputProperties=FEATURES)

# Clasificar la imagen
classified_rf = image_std.classify(rf).clip(REGION)

show_classification(classified_rf, "Random Forest")

**Compare los tres mapas.** ¿Dónde coinciden? ¿Dónde difieren - la orilla, el borde
del pueblo, las colinas? Las diferencias muestran dónde las clases se **confunden espectralmente**.

---

## 1.7 - Evaluación de la exactitud del Random Forest

Clasificamos los píxeles de **prueba** - nunca vistos en el entrenamiento - y comparamos la predicción con la
clase verdadera. `errorMatrix(true, predicted, order)` construye la matriz de confusión.

In [ ]:
# Clasificar la tabla de prueba y construir la matriz de confusión
test_rf = test.classify(rf)
cm = test_rf.errorMatrix(CLASS_PROPERTY, "classification", CLASS_CODES)

print("Overall accuracy:", round(cm.accuracy().getInfo(), 3))
print("Kappa           :", round(cm.kappa().getInfo(), 3))

In [ ]:
# Matriz de confusión como DataFrame: filas = referencia (prueba), columnas = clasificación
df_cm = pd.DataFrame(cm.array().getInfo(), index=CLASS_NAMES, columns=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Confusion matrix - Random Forest (test samples)")
ax.set_xlabel("Classification (predicted)")
ax.set_ylabel("Reference (test samples)")
plt.show()

In [ ]:
# Exactitud del productor y del consumidor por clase, y los errores correspondientes
df_acc = pd.DataFrame({
    "Producer's accuracy": cm.producersAccuracy().project([0]).getInfo(),   # por fila
    "Consumer's accuracy": cm.consumersAccuracy().project([1]).getInfo(),   # por columna
}, index=CLASS_NAMES)

df_acc["Omission error"] = 1 - df_acc["Producer's accuracy"]
df_acc["Commission error"] = 1 - df_acc["Consumer's accuracy"]
df_acc.round(3)

In [ ]:
# Exactitud del productor vs del consumidor por clase
ax = df_acc[["Producer's accuracy", "Consumer's accuracy"]].plot.bar(
    figsize=(8, 4), color=["#4c72b0", "#dd8452"], rot=0)
ax.set_title("Accuracy per class - Random Forest")
ax.set_xlabel("Class")
ax.set_ylabel("Accuracy (0-1)")
ax.set_ylim(0, 1)
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**Cómo leerla.** Una clase con **baja exactitud del productor** se está *perdiendo* (omisión): sus
píxeles terminan en otra clase. Una clase con **baja exactitud del consumidor** está *recibiendo* píxeles
de otras clases (comisión): el mapa muestra más de ella de lo que realmente hay. Busque la celda
fuera de la diagonal con el número más grande - ese es el par de clases que más se confunde.

### Los tres clasificadores lado a lado

El mismo conjunto de prueba, la misma métrica:

In [ ]:
# Exactitud global de cada clasificador sobre las mismas muestras de prueba
classifiers = {"CART": cart, "SVM": svm, "Random Forest": rf}

overall = {name: test.classify(c).errorMatrix(CLASS_PROPERTY, "classification", CLASS_CODES)
                     .accuracy().getInfo()
           for name, c in classifiers.items()}

pd.DataFrame({"Overall accuracy": overall}).round(3)

---

## 1.8 - Importancia de las variables (Random Forest)

`rf.explain()` devuelve un diccionario con información sobre el modelo entrenado, incluida la
**importancia** de cada variable: cuánto ayudó a los árboles a separar las clases. La convertimos
a porcentaje y la graficamos.

In [ ]:
# Importancia de cada variable, a partir del Random Forest entrenado
importance = ee.Dictionary(rf.explain()).get("importance").getInfo()

# A una Series de pandas, en porcentaje, ordenada
df_imp = pd.Series(importance)
df_imp = (100 * df_imp / df_imp.sum()).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(df_imp.index, df_imp.values, color="#2e8b57")
ax.set_title("Feature importance - Random Forest")
ax.set_xlabel("Relative importance (%)")
ax.set_ylabel("Feature")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**Preguntas.** ¿Los índices son más importantes que las bandas originales? ¿Ayudaron la **elevación** o la
**pendiente**? Las variables con importancia muy baja se pueden eliminar - un modelo más simple es más rápido
y a menudo igual de exacto.

---

## 1.9 - Ejercicio

1. **K-Means.** Cambie k a 3 y a 8. ¿Qué valor da clústeres más cercanos a nuestras cinco
   clases?
2. **Sus muestras.** Recolecte nuevas muestras en el Sample Collector para la clase con la
   **menor exactitud del productor**, exporte de nuevo el GeoJSON y vuelva a ejecutar las secciones 1.6 y 1.7.
   ¿Mejoró la exactitud?
3. **Hiperparámetros.** Entrene el Random Forest con 10, 50 y 200 árboles, y el SVM con
   `gamma` 0.01, 0.1 y 1. Haga una pequeña tabla con la exactitud global de cada uno.
4. **Menos variables.** Conserve solo las 6 variables más importantes de la sección 1.8 y entrene
   de nuevo el Random Forest. ¿Qué pasa con la exactitud?

**Entregable**: el notebook con los mapas, las tablas de las preguntas 3 y 4, y un
párrafo corto explicando qué clasificador elegiría y por qué.

---

## Referencias

**Aprendizaje automático y clasificación**

- Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5-32.
- Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow*. O'Reilly.
- VanderPlas, J. (2016). *Python Data Science Handbook*. O'Reilly.
  <https://jakevdp.github.io/PythonDataScienceHandbook/05.01-what-is-machine-learning.html>
- Zanotta, D., Zortea, M., Ferreira, M. P. (2019). *Processamento de imagens de satélite*.
  São Paulo: Oficina de Textos.
- Jensen, J. R. (2005). *Introductory Digital Image Processing: A Remote Sensing Perspective*.
  Prentice Hall.
- Belsare, K., Singh, M. (2022). Machine learning approaches and the Internet of Things.
- Zhuhadar, L. P., Lytras, M. D. (2023). The application of AutoML techniques in diabetes
  diagnosis. *Sustainability*, 15(18), 13484. <https://doi.org/10.3390/su151813484>
- Jiang, D. et al. (2013). *Remote Sensing*, 5, 3156-3171. <https://doi.org/10.3390/rs5073156>
- Escovedo, T., Koshiyama, A. (2020). *Introdução a Data Science*. Casa do Código.
  <https://tatianaesc.medium.com/machine-learning-conceitos-e-modelos-f0373bf4f445>
- Souza, C. M. et al. (2020). Reconstructing three decades of land use and land cover changes in
  Brazilian biomes with Landsat archive and Earth Engine. *Remote Sensing*, 12(17), 2735.
  <https://doi.org/10.3390/rs12172735>
- Jordan, M. I., Mitchell, T. M. (2015). Machine learning: Trends, perspectives, and prospects.
  *Science*, 349(6245), 255-260. <https://doi.org/10.1126/science.aaa8415>
- Janiesch, C., Zschech, P., Heinrich, K. (2021). Machine learning and deep learning.
  *Electronic Markets*, 31, 685-695. <https://doi.org/10.1007/s12525-021-00475-2>
- Sarker, I. H. (2021). Machine learning: Algorithms, real-world applications and research
  directions. *SN Computer Science*, 2, 160. <https://doi.org/10.1007/s42979-021-00592-x>
- Maxwell, A. E., Warner, T. A., Fang, F. (2018). Implementation of machine-learning
  classification in remote sensing: an applied review. *International Journal of Remote
  Sensing*, 39(9), 2784-2817. <https://doi.org/10.1080/01431161.2018.1433343>

**Google Earth Engine**

- Gandhi, U. (2021). *End-to-End Google Earth Engine* (Supervised classification, feature
  importance, hyperparameter tuning). <https://courses.spatialthoughts.com/end-to-end-gee.html>
- Clasificación supervisada: <https://developers.google.com/earth-engine/guides/classification>
- Clasificación no supervisada: <https://developers.google.com/earth-engine/guides/clustering>
- NASADEM: <https://developers.google.com/earth-engine/datasets/catalog/NASA_NASADEM_HGT_001>
- ESA WorldCover: <https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200>
- geemap: <https://geemap.org>